# Models

> Data types for job monitor URLs, configuration, and resource snapshots.

In [ ]:
#| default_exp models

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from typing import Optional

## JobMonitorUrls

URL endpoints generated by the route factory.

In [ ]:
#| export
@dataclass
class JobMonitorUrls:
    """URL endpoints for the job monitor routes."""
    trigger: str   # POST -- submit job
    progress: str  # GET -- poll progress
    cancel: str    # POST -- cancel job

In [ ]:
# Test JobMonitorUrls
urls = JobMonitorUrls(
    trigger="/workflow/seg/fa/trigger",
    progress="/workflow/seg/fa/progress",
    cancel="/workflow/seg/fa/cancel",
)
assert urls.trigger == "/workflow/seg/fa/trigger"
assert urls.progress == "/workflow/seg/fa/progress"
assert urls.cancel == "/workflow/seg/fa/cancel"
print(f"URLs: trigger={urls.trigger}, progress={urls.progress}, cancel={urls.cancel}")

URLs: trigger=/workflow/seg/fa/trigger, progress=/workflow/seg/fa/progress, cancel=/workflow/seg/fa/cancel


## ResourceSnapshot

Point-in-time resource usage for a plugin worker process.

In [ ]:
#| export
@dataclass
class ResourceSnapshot:
    """Point-in-time resource usage for a worker."""
    worker_pid: int                            # Worker process ID
    cpu_percent: float                         # CPU utilization %
    memory_rss_mb: float                       # Process tree RSS in MB
    gpu_memory_mb: Optional[float] = None      # Per-process GPU memory in MB
    gpu_index: Optional[int] = None            # Which GPU device
    gpu_name: Optional[str] = None             # GPU device name
    gpu_total_mb: Optional[float] = None       # Total GPU memory in MB
    gpu_load_percent: Optional[float] = None   # GPU compute utilization %

In [ ]:
# Test ResourceSnapshot -- CPU/RAM only
snap = ResourceSnapshot(worker_pid=12345, cpu_percent=45.2, memory_rss_mb=1200.5)
assert snap.worker_pid == 12345
assert snap.cpu_percent == 45.2
assert snap.memory_rss_mb == 1200.5
assert snap.gpu_memory_mb is None
assert snap.gpu_index is None
print(f"CPU-only snapshot: PID={snap.worker_pid}, CPU={snap.cpu_percent}%, RAM={snap.memory_rss_mb}MB")

CPU-only snapshot: PID=12345, CPU=45.2%, RAM=1200.5MB


In [ ]:
# Test ResourceSnapshot -- with GPU
snap_gpu = ResourceSnapshot(
    worker_pid=12345,
    cpu_percent=45.2,
    memory_rss_mb=1200.5,
    gpu_memory_mb=2048.0,
    gpu_index=0,
    gpu_name="NVIDIA GeForce RTX 4090",
    gpu_total_mb=24564.0,
    gpu_load_percent=72.0,
)
assert snap_gpu.gpu_memory_mb == 2048.0
assert snap_gpu.gpu_name == "NVIDIA GeForce RTX 4090"
assert snap_gpu.gpu_total_mb == 24564.0
print(f"GPU snapshot: {snap_gpu.gpu_memory_mb}MB / {snap_gpu.gpu_total_mb}MB on {snap_gpu.gpu_name}")

GPU snapshot: 2048.0MB / 24564.0MB on NVIDIA GeForce RTX 4090


In [ ]:
# Test serialization round-trip
snap_dict = asdict(snap_gpu)
assert snap_dict['worker_pid'] == 12345
assert snap_dict['gpu_memory_mb'] == 2048.0
assert snap_dict['gpu_name'] == "NVIDIA GeForce RTX 4090"
snap_roundtrip = ResourceSnapshot(**snap_dict)
assert snap_roundtrip == snap_gpu
print("Serialization round-trip passed!")

Serialization round-trip passed!


## JobMonitorConfig

Display configuration for a job monitor instance.

In [ ]:
#| export
@dataclass
class JobMonitorConfig:
    """Configuration for a job monitor instance."""
    modal_title: str = "Job Execution"          # Modal header title
    trigger_label: str = "Run"                  # Trigger button label
    trigger_icon: Optional[str] = None          # Lucide icon name for trigger button
    progress_label: str = "View Progress"       # Progress button label (when modal closed)
    sse_interval_s: float = 0.5                 # Server-side SSE push interval in seconds
    log_lines: int = 50                         # Number of log lines to show
    overlay_z_index: int = 10                   # Overlay z-index

In [ ]:
# Test JobMonitorConfig defaults
config = JobMonitorConfig()
assert config.modal_title == "Job Execution"
assert config.trigger_label == "Run"
assert config.trigger_icon is None
assert config.progress_label == "View Progress"
assert config.sse_interval_s == 0.5
assert config.log_lines == 50
assert config.overlay_z_index == 10
print(f"Default config: title='{config.modal_title}', trigger='{config.trigger_label}', sse_interval={config.sse_interval_s}s")

In [ ]:
# Test JobMonitorConfig custom values
fa_config = JobMonitorConfig(
    modal_title="Force Alignment",
    trigger_label="Force Align",
    trigger_icon="audio-waveform",
    sse_interval_s=0.3,
    log_lines=80,
)
assert fa_config.modal_title == "Force Alignment"
assert fa_config.trigger_icon == "audio-waveform"
assert fa_config.sse_interval_s == 0.3
assert fa_config.log_lines == 80
print(f"FA config: title='{fa_config.modal_title}', icon='{fa_config.trigger_icon}', lines={fa_config.log_lines}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()